In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Pen's Code

Importing Data

In [ ]:
DF_visit = pd.read_csv('Visit data.csv') #visitor data from 2015-2024
DF_econ = pd.read_csv("Economic data.csv") #economic data from 2015-2024
states = pd.read_csv("states.csv") #regional breakdown of states
states = states.drop(columns = "State") #Removing column to make merging easier later
longlat_bad = pd.read_csv("df_2.csv") #Location data for the national parks

Cleaning and wrangling Visitor Data

In [ ]:
DF_visit.head()
DF_visit["State"] = DF_visit["State"].str.strip() # Maine (ME) had an extra space, which caused problems later

In [ ]:
DF_visit['Month_Num'] = pd.to_datetime(DF_visit['Month'], format='%B').dt.month
DF_visit['Date'] = pd.to_datetime(DF_visit.Year.astype(str) + '/' + DF_visit.Month_Num.astype(str) + '/01') #making date-time for time series visualizations

DF_visit["Join Code"] = DF_visit["Park Code"] + DF_visit["Year"].astype(str) #combining year and park code

In [ ]:
DF_visit = pd.merge(DF_visit, states, left_on = "State", right_on = "State Code",how = "left" ) #merging visitor data with regional state breakdown

In [ ]:
DF_visit = DF_visit.drop(columns = "State Code") #duplicate column

#To make merging with location data set easier
DF_visit["Park Name"] = DF_visit["Park Name"].str.replace(" NP", " National Park")
DF_visit["Park Name Only"] = DF_visit["Park Name"].str.split(" National Park").str[0]
DF_visit["Park Name Only"] = DF_visit["Park Name Only"].str.split(" &").str[0]
DF_visit["Park Name Only"] = DF_visit["Park Name Only"].str.replace("Hawaii", "Hawaiʻi")
DF_visit["Park Name Only"] = DF_visit["Park Name Only"].str.strip()

Visitor Data by year instead of month

In [ ]:
#making a dictionary for how to handle columns when summing months in a year

keys_bad = DF_visit.columns.tolist()
to_remove = ["Month","Month_Num","Date"]
keys= list(filter(lambda x: x not in to_remove, keys_bad))

values = ["first","first","first","first","sum","sum","sum","sum","sum","sum","sum","sum","sum","sum","sum","sum","first","first","first","first"]

agg_list = dict(zip(keys,values))

In [8]:
DF_visit_by_year = DF_visit.groupby(by=["Park Code","Year"]).agg(agg_list)

Cleaning Economic Data

In [ ]:
DF_econ["Join Code"] = DF_econ["Code"] + DF_econ["Year"].astype(str) # to make merging easier
DF_econ = DF_econ.drop(columns= ["Year","Code"]) #getting rid of duplicate columns

Merging visitor and economic data

In [ ]:
DF_all = pd.merge(DF_visit_by_year, DF_econ, on = "Join Code")
DF_all = DF_all.drop(columns= ["Name", "Recreation Visits"]) #Dropping duplicate columns

Location Data cleaning and wrangling

In [ ]:
longlat_mid = longlat_bad.drop(columns = [ "Unnamed: 0","Image", "Description"]) #Dropping irrelevant columns
longlat = longlat_mid.drop([1,37,44], axis = 0) #dropping rows that didnt matter to us

longlat["Name"] = longlat["Name"].str.replace('Wrangell–St.\xa0Elias ', "Wrangell - St. Elias") #name replacing for merge
longlat["Name"] = longlat["Name"].str.strip()

#Extracting longitude and latitude from Location column
longlat["LongLat"] = longlat["Location"].str.split("/").str[1]
longlat["LAT"] = longlat["LongLat"].str.split("°").str[0]
longlat["LAT"] = longlat["LAT"].str.split("\ufeff").str[1]
longlat["LONG"] = longlat["LongLat"].str.split("°N ").str[1].str.split("°W").str[0]
longlat["LONG"] = "-" + longlat["LONG"]
longlat["LONG"] = longlat["LONG"].astype(float)
longlat["LAT"] = longlat["LAT"].astype(float)

longlat = longlat.drop(columns = ["LongLat", "Location", "Recreation visitors (2021)[11]"]) #dropping duplicate columns

Adding location to datasets

In [ ]:
Visit_longlat = pd.merge(DF_visit, longlat, left_on = "Park Name Only", right_on= "Name", how = "left") #merging visitor and location data
Visit_econ_longlat = pd.merge(DF_all, longlat, left_on = "Park Name Only", right_on= "Name", how = "left") #merging visitor and economic data with location data

Exporting

In [13]:
Visit_longlat.to_csv("Visit Location Data.csv")
Visit_econ_longlat.to_csv("Visit Econ Location Data.csv")